# Projeto: Coleta de Previsão Climática – INPE

### Descrição Geral
Este documento descreve o processo de coleta de dados meteorológicos via API pública [INPE – CPTEC](https://www.cptec.inpe.br/), utilizando o ambiente Databricks.<br>
O objetivo é extrair previsões climáticas para todas as capitais brasileiras, incluindo a previsão do dia atual e dos próximos dias, estruturando as informações para posterior análise em pipelines de dados.

---

### Informações Técnicas

- **Fonte de Dados**: [INPE – CPTEC](https://www.cptec.inpe.br/)
- **Tecnologias Utilizadas**: 
  - Python
  - Apache Spark (PySpark)
  - Databricks Notebooks

- **Dados Coletados**:
  - Cidade e Unidade Federativa (UF)
  - Data da previsão
  - Temperatura mínima e máxima
  - Probabilidade de chuva
  - Fonte da informação

---

### Histórico de Atualizações

| Data       | Responsável      | Versão | Descrição                                                                 |
|------------|------------------|--------|---------------------------------------------------------------------------|
| 12/05/2025 | Evelise Dias       | 1.0    | Estrutura inicial do projeto e teste de viabilidade                                           |
| 13/05/2025 | Evelise Dias      | 1.1    | Refatoração da lógica de coleta e organização do script |
| 16/05/2025 | Evelise Dias      | 1.2    | Criação das docstrings para cada função e atualização do sumário |

---



In [0]:
%run ../utils/utils_run

In [0]:
%run ../utils/funcoes_coleta_dados/funcoes_inpe_webscraping

In [0]:
%run ../utils/funcoes_coleta_dados/funcoes_comuns_motores


### Função principal que coleta previsões
Esta função orquestra a coleta das previsões para todas as capitais, usando funções helpers criadas no notebook `funções_inpe_webscraping` para montar URLs e extrair dados, garantindo que cada etapa fique clara e modular. Ela centraliza a lógica do fluxo de dados e agrega informações locais para que o resultado seja uma lista pronta para análise.

In [0]:
"""
Gera um DataFrame Spark com as previsões do tempo para todas as capitais do enum informado.

Args:
    nome_enum (str): Nome do enum que contém as capitais e seus estados (UF).

Returns:
    DataFrame: DataFrame Spark contendo as previsões do tempo para as capitais, com colunas 
               como data, temperaturas, precipitação, fonte, estado e cidade.
"""


df_previsoes = obter_previsoes_todas_capitais_em_df("CAPITAIS_ID_CLIMA_TEMPO")


In [0]:
df_previsoes_df = spark.createDataFrame(df_previsoes)


In [0]:
df_previsoes_df = adicionaColunasDeCarga(df_previsoes_df)

In [0]:
"""
Salva um DataFrame Spark no Data Lake na pasta especificada, no formato desejado.

Parameters:
    df (DataFrame): DataFrame Spark a ser salvo.
    caminho (str): Caminho da pasta no Data Lake onde o arquivo será salvo.
    format (str): Formato do arquivo (ex: "parquet", "csv")

Returns:
    None
"""

salvar_no_datalake(df_previsoes_df, f"{landingzone}/previsoes_inpe", format= "parquet")